In [1]:

import requests
import os
from dotenv import load_dotenv

load_dotenv("credentials.env", override=True)

client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

if not client_id or not client_secret:
    raise ValueError("Credentials not found!")

token_url = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

response = requests.post(
    token_url,
    headers={
        "Content-Type": "application/x-www-form-urlencoded"
    },
    data={
        "grant_type": "client_credentials",
        "client_id": client_id.strip(),
        "client_secret": client_secret.strip()
    },
    timeout=30
)

print("Status Code:", response.status_code)

if response.status_code == 200:
    access_token = response.json()["access_token"]
    print("Authentication Successful!")
else:
    print(response.json())


Status Code: 200
Authentication Successful!


In [2]:

import requests
import pandas as pd

# API endpoint
url = "https://opensky-network.org/api/states/all"

# Authentication header
headers = {
    "Authorization": f"Bearer {access_token}"
}

# Fetch flight data
response = requests.get(
    url,
    headers=headers,
    timeout=30
)

if response.status_code == 200:

    flight_data = response.json()

    print("Data fetched successfully!")

    print("API Timestamp:", flight_data["time"])

    states = flight_data["states"]

    if states:
        df = pd.DataFrame(states)

        print("\nTotal Aircraft:", len(df))

        print("\nFirst 5 Records:")
        print(df.head())

    else:
        print("No aircraft data available.")

else:
    print("Error:", response.status_code)
    print(response.text)


Data fetched successfully!
API Timestamp: 1792871680

Total Aircraft: 11427

First 5 Records:
       0         1              2             3           4         5   \
0  a5f852  RTY484    United States  1.792872e+09  1792871680 -104.4636   
1  a09281  N136LM    United States  1.792872e+09  1792871679  -84.4754   
2  ae1fa0  SHADO29   United States  1.792872e+09  1792871539  -84.9797   
3  80162d  AXB1818           India  1.792872e+09  1792871680   75.0513   
4  39de4b  TVF8603          France  1.792872e+09  1792871680    5.0609   

        6         7      8       9       10    11    12        13    14  \
0  40.6167   2148.84  False   63.74  205.32 -1.30  None   2232.66  1200   
1  34.2663   1005.84  False   50.95  234.87  3.90  None   1104.90   NaN   
2  38.3772    251.46  False   34.21  282.15 -0.98  None    365.76   NaN   
3  26.3914  10805.16  False  264.04   18.05 -4.23  None  11254.74   NaN   
4  44.1229  11582.40  False  186.66  335.06  0.00  None  11971.02  1000   

      15  

In [ ]:
# OpenSky API ke states response mein 17 columns hain. Abhi unke names numeric indexes hain
columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source"
]

df = pd.DataFrame(states, columns=columns)

print(df.head())
print(df.columns)


   icao24  callsign origin_country  time_position  last_contact  longitude  \
0  a5f852  RTY484    United States   1.792872e+09    1792871680  -104.4636   
1  a09281  N136LM    United States   1.792872e+09    1792871679   -84.4754   
2  ae1fa0  SHADO29   United States   1.792872e+09    1792871539   -84.9797   
3  80162d  AXB1818           India   1.792872e+09    1792871680    75.0513   
4  39de4b  TVF8603          France   1.792872e+09    1792871680     5.0609   

   latitude  baro_altitude  on_ground  velocity  true_track  vertical_rate  \
0   40.6167        2148.84      False     63.74      205.32          -1.30   
1   34.2663        1005.84      False     50.95      234.87           3.90   
2   38.3772         251.46      False     34.21      282.15          -0.98   
3   26.3914       10805.16      False    264.04       18.05          -4.23   
4   44.1229       11582.40      False    186.66      335.06           0.00   

  sensors  geo_altitude squawk    spi  position_source  
0    

In [ ]:
# Understand your dataset
print(df.shape)

print(df.info())

print(df.isnull().sum())

print(df.describe())

(11427, 17)
<class 'pandas.DataFrame'>
RangeIndex: 11427 entries, 0 to 11426
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   icao24           11427 non-null  str    
 1   callsign         11427 non-null  str    
 2   origin_country   11427 non-null  str    
 3   time_position    11364 non-null  float64
 4   last_contact     11427 non-null  int64  
 5   longitude        11364 non-null  float64
 6   latitude         11364 non-null  float64
 7   baro_altitude    10376 non-null  float64
 8   on_ground        11427 non-null  bool   
 9   velocity         11425 non-null  float64
 10  true_track       11427 non-null  float64
 11  vertical_rate    10402 non-null  float64
 12  sensors          0 non-null      object 
 13  geo_altitude     10298 non-null  float64
 14  squawk           6640 non-null   str    
 15  spi              11427 non-null  bool   
 16  position_source  11427 non-null  int64  
dtypes: bool(2),

In [ ]:
df['sensors'].isnull().sum()
# Is column mein 11,427 missing values hain.
# Iska matlab API response mein sensor information available nahi hai. Is project ke current scope mein ise remove kar sakte hain.
# 63 records mein latitude aur longitude missing hain.
# baro_altitude aur geo_altitude mein missing values hain.
# Squawk missing hona necessarily data error nahi hai. Isliye ise preserve karenge.

np.int64(11427)

In [ ]:
print(df.duplicated().sum())
# Check duplicate records

0


In [ ]:
missing_percentage = (
    df.isnull().sum() / len(df)
) * 100

print(missing_percentage.sort_values(ascending=False))
# Check missing values percentage

sensors            100.000000
squawk              41.892010
geo_altitude         9.880109
baro_altitude        9.197515
vertical_rate        8.969983
longitude            0.551326
time_position        0.551326
latitude             0.551326
velocity             0.017502
icao24               0.000000
callsign             0.000000
origin_country       0.000000
last_contact         0.000000
on_ground            0.000000
true_track           0.000000
spi                  0.000000
position_source      0.000000
dtype: float64


In [ ]:
print((df['callsign'].str.strip() == '').sum())
# Check blank callsigns

158


In [ ]:
print("Total Countries:", df['origin_country'].nunique())

print(df['origin_country'].value_counts().head(10)) 
# Check unique countries

Total Countries: 107
origin_country
United States     6790
Canada             476
United Kingdom     384
Turkey             313
Ireland            259
Malta              226
Germany            207
Spain              170
France             158
China              139
Name: count, dtype: int64


In [11]:
df.drop(columns=['sensors'], inplace=True)
# It contains 100% missing values, so it provides no useful information in this snapshot

In [13]:
# Some callsigns contain empty spaces rather than actual values. Convert those blanks into proper missing values:
df['callsign'] = df['callsign'].str.strip() 
df['callsign'] = df['callsign'].replace('', pd.NA)

In [14]:
print(df.isnull().sum())
print(df.shape)

icao24                0
callsign            158
origin_country        0
time_position        63
last_contact          0
longitude            63
latitude             63
baro_altitude      1051
on_ground             0
velocity              2
true_track            0
vertical_rate      1025
geo_altitude       1129
squawk             4787
spi                   0
position_source       0
dtype: int64
(11427, 16)


In [ ]:
# Check aircraft currently on the ground
print(df['on_ground'].value_counts())

# Approximately 91% of the aircraft are flying and 9% are on the ground.

on_ground
False    10419
True      1008
Name: count, dtype: int64


In [ ]:
# # Check missing coordinates
print(df[df['longitude'].isnull()][['icao24', 'callsign', 'origin_country', 'on_ground']].head(10))
# on_ground tells us whether an aircraft is on the ground.
# longitude and latitude identify an aircraft's geographic position.
# Some aircraft may not have position information in the current API snapshot.
# Your sample shows that some aircraft have missing callsigns, but their on_ground value is False.
# This means we should not assume that missing coordinates mean the aircraft is on the ground.
# The API may not have a recent position available for those aircraft.

      icao24 callsign origin_country  on_ground
598   ab4b30      NaN  United States      False
649   e0b14c      NaN      Argentina      False
666   d005ee      NaN                     False
739   4bc884  PGT2385         Turkey      False
795   ab0104      NaN  United States      False
1217  a03c22      NaN  United States      False
1563  ab4db1      NaN  United States      False
1784  a1916a      NaN  United States      False
1800  ada8b7      NaN  United States      False
1860  a6f737     SCU7  United States      False


In [ ]:
# Check missing coordinates by ground status
print(
    df.groupby('on_ground')[['longitude', 'latitude']]
      .apply(lambda x: x.isnull().any(axis=1).sum())
)
# This will count how many aircraft have missing coordinates in each ground-status group.

on_ground
False    62
True      1
dtype: int64


In [20]:
print(df[['longitude', 'latitude']].isnull().sum())

longitude    63
latitude     63
dtype: int64


In [21]:
print(df[['baro_altitude', 'geo_altitude']].describe())

       baro_altitude  geo_altitude
count   10376.000000  10298.000000
mean     6274.928498   6507.305787
std      4551.137789   4685.727663
min      -175.260000    -38.100000
25%      1363.980000   1447.800000
50%      6995.160000   7200.900000
75%     10668.000000  11079.480000
max     14630.400000  15072.360000


In [22]:
print(df.groupby('on_ground')[['baro_altitude', 'geo_altitude']].mean())

           baro_altitude  geo_altitude
on_ground                             
False        6285.190629   6507.305787
True          680.987368           NaN


In [23]:
print("Unique Squawk Codes:", df['squawk'].nunique())

print(df['squawk'].value_counts().head(10))

print("Missing Squawk Values:", df['squawk'].isnull().sum())

Unique Squawk Codes: 3088
squawk
1000    575
1200    302
6037      8
1202      7
6211      7
1376      7
1356      6
1346      6
3207      6
2274      6
Name: count, dtype: int64
Missing Squawk Values: 4787


In [24]:
print(df.dtypes)

icao24                 str
callsign               str
origin_country         str
time_position      float64
last_contact         int64
longitude          float64
latitude           float64
baro_altitude      float64
on_ground             bool
velocity           float64
true_track         float64
vertical_rate      float64
geo_altitude       float64
squawk                 str
spi                   bool
position_source      int64
dtype: object


In [25]:
print(df[['time_position', 'last_contact', 'position_source']].head())

   time_position  last_contact  position_source
0   1.792872e+09    1792871680                0
1   1.792872e+09    1792871679                0
2   1.792872e+09    1792871539                0
3   1.792872e+09    1792871680                0
4   1.792872e+09    1792871680                0


In [ ]:
df['time_position'] = pd.to_datetime(
    df['time_position'], unit='s', errors='coerce', utc=True
)

df['last_contact'] = pd.to_datetime(
    df['last_contact'], unit='s', errors='coerce', utc=True
)
# We are using UTC to preserve the original API timestamp timezone

In [27]:
print(df[['time_position', 'last_contact']].head())
print(df[['time_position', 'last_contact']].dtypes)

              time_position              last_contact
0 2026-10-24 19:54:40+00:00 2026-10-24 19:54:40+00:00
1 2026-10-24 19:54:38+00:00 2026-10-24 19:54:39+00:00
2 2026-10-24 19:52:19+00:00 2026-10-24 19:52:19+00:00
3 2026-10-24 19:54:38+00:00 2026-10-24 19:54:40+00:00
4 2026-10-24 19:54:40+00:00 2026-10-24 19:54:40+00:00
time_position    datetime64[s, UTC]
last_contact     datetime64[s, UTC]
dtype: object


In [32]:
df['time_position'] = pd.to_datetime(
    df['time_position'], unit='s', errors='coerce', utc=True
)

df['last_contact'] = pd.to_datetime(
    df['last_contact'], unit='s', errors='coerce', utc=True
)
# time_position has missing values, so errors='coerce' will preserve them as NaT rather than causing an error.

In [33]:
print(df[['time_position', 'last_contact']].head())
print(df[['time_position', 'last_contact']].dtypes)

              time_position              last_contact
0 2026-10-24 19:54:40+00:00 2026-10-24 19:54:40+00:00
1 2026-10-24 19:54:38+00:00 2026-10-24 19:54:39+00:00
2 2026-10-24 19:52:19+00:00 2026-10-24 19:52:19+00:00
3 2026-10-24 19:54:38+00:00 2026-10-24 19:54:40+00:00
4 2026-10-24 19:54:40+00:00 2026-10-24 19:54:40+00:00
time_position    datetime64[s, UTC]
last_contact     datetime64[s, UTC]
dtype: object


In [ ]:
# Check the remaining numerical columns
# Before exporting our cleaned dataset, let's check whether any impossible or unusual values exist/.
print(df[['velocity', 'true_track', 'vertical_rate',
          'baro_altitude', 'geo_altitude']].describe())
# velocity: Aircraft speed in metres per second.
# true_track: Direction of travel in degrees.
# vertical_rate: Rate of climb or descent in metres per second.
# baro_altitude and geo_altitude: Aircraft altitude in metres.

           velocity    true_track  vertical_rate  baro_altitude  geo_altitude
count  11425.000000  11427.000000   10402.000000   10376.000000  10298.000000
mean     148.681990    178.507796      -0.213730    6274.928498   6507.305787
std       94.918859    104.982701       4.956813    4551.137789   4685.727663
min        0.000000      0.000000     -28.290000    -175.260000    -38.100000
25%       57.100000     88.925000      -0.980000    1363.980000   1447.800000
50%      174.370000    176.290000       0.000000    6995.160000   7200.900000
75%      229.480000    271.550000       0.330000   10668.000000  11079.480000
max     2089.180000    462.000000     124.850000   14630.400000  15072.360000


In [35]:
# Find invalid track directions
print(df[df['true_track'] > 360][
    ['icao24', 'callsign', 'true_track', 'velocity']
])

     icao24 callsign  true_track  velocity
666  d005ee      NaN       462.0      30.6


In [ ]:
# In data engineering, cleaning doesn't mean blindly deleting outliers. We first identify suspicious records, investigate them, and then decide whether to correct, retain, or exclude them.
# Find unusually high vertical rates
print(df[df['vertical_rate'].abs() > 20][
    ['icao24', 'callsign', 'vertical_rate']
])

       icao24 callsign  vertical_rate
171    ae61e3  SNIPR10          28.61
666    d005ee      NaN          31.20
1286   a9d9a3   N7338C         122.25
1965   4ac9eb  SAS7470         -28.29
2704   a9555f   N700JJ         -25.36
2915   7c77fc   QFA151          23.73
2940   ac6d30    N90DT          52.02
3517   a9973c   JTL717          20.16
4260   abeeea  SWA3025          20.16
4662   a3fbad    N356G          20.16
4764   abbd8d  SWA3235          20.16
5098   a014b9   N104MX          81.93
5190   344696  VLG2018          25.03
5252   c053b4   ASP655          20.81
5593   a8b5b2   JTZ433          20.48
5679   a4ab32   CITT52         -28.29
5992   a0a6ca   N141DZ         -23.08
6073   4690e2  AEE6125          24.71
7124   781008  CCA720D         124.85
7698   c061a2   POE125          20.81
8586   a1d18b    YEL16          20.48
8662   c003a8   JZA748          20.81
8810   c05dd8   JZA890          20.16
9722   a455c7  FFT3871          20.48
10362  a1b098   N208JW         -20.16
10689  7c7a4

In [38]:
df.loc[df['true_track'] > 360, 'true_track'] = pd.NA

In [39]:

print(df['true_track'].describe())

print("Invalid Track Values:",
      (df['true_track'] > 360).sum())

count    11426.000000
mean       178.482985
std        104.953785
min          0.000000
25%         88.922500
50%        176.235000
75%        271.547500
max        359.860000
Name: true_track, dtype: float64
Invalid Track Values: 0


In [40]:
print(df['vertical_rate'].describe(percentiles=[0.90, 0.95, 0.99]))

count    10402.000000
mean        -0.213730
std          4.956813
min        -28.290000
90%          4.550000
95%          8.130000
99%         14.630000
max        124.850000
Name: vertical_rate, dtype: float64


In [ ]:
print(df['vertical_rate'].sort_values(ascending=False).head(10))
# vertical_rate is measured in metres per second.
# Positive values indicate climbing.
# Negative values indicate descending.
# We want to identify extreme values without deleting legitimate observations.

7124     124.85
1286     122.25
5098      81.93
2940      52.02
666       31.20
171       28.61
5190      25.03
6073      24.71
2915      23.73
11204     23.08
Name: vertical_rate, dtype: float64


In [42]:
# This column contains numeric codes that identify the source of aircraft position information.
print(df['position_source'].value_counts())

position_source
0    11427
Name: count, dtype: int64


In [43]:
print(df['position_source'].nunique())

1


In [44]:
df.drop(columns=['position_source'], inplace=True)

In [45]:
print(df.shape)
print(df.columns)

(11427, 15)
Index(['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact',
       'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity',
       'true_track', 'vertical_rate', 'geo_altitude', 'squawk', 'spi'],
      dtype='str')


In [46]:
# Final dataset verification
print("Final Shape:", df.shape)
print(df.columns.tolist())

# Export cleaned dataset
df.to_csv("cleaned_flight_data.csv", index=False)

print("Cleaned CSV exported successfully!")

Final Shape: (11427, 15)
['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'geo_altitude', 'squawk', 'spi']
Cleaned CSV exported successfully!
